# 01 · Simple Pipeline Demo

This notebook shows a very small NL-to-SQL example.

Goal:
1. turn a schema into prompt text
2. build a few-shot prompt
3. clean noisy model output
4. validate the final SQL

It does **not** call a real model or a real database. It is only a simple walkthrough.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "nl2sql").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("working directory:", PROJECT_ROOT)


In [ ]:
from nl2sql.agent.agent_tools import schema_to_text
from nl2sql.core.prompting import make_few_shot_messages
from nl2sql.core.llm import extract_first_select
from nl2sql.core.sql_guardrails import clean_candidate_with_reason
from nl2sql.core.postprocess import guarded_postprocess
from nl2sql.core.validation import validate_sql

print("imports ok")


## 1) Example Schema
We use a tiny fake schema so the demo stays easy to read.

In [ ]:
schema_cache = {
    "tables": [
        {"name": "customers", "columns": [{"name": "customerNumber"}, {"name": "customerName"}, {"name": "country"}]},
        {"name": "orders", "columns": [{"name": "orderNumber"}, {"name": "customerNumber"}, {"name": "orderDate"}]},
    ],
    "foreign_keys": [
        {"table": "orders", "column": "customerNumber", "ref_table": "customers", "ref_column": "customerNumber"},
    ],
}

schema_text = schema_to_text(schema_cache)
print(schema_text)


## 2) Build a Prompt
The prompt contains the schema, a few examples, and the new question.

In [ ]:
nlq = "Which countries have the most orders?"

exemplars = [
    {"nlq": "List customer names in France.", "sql": "SELECT customerName FROM customers WHERE country = 'France';"},
    {"nlq": "Count orders for each customer.", "sql": "SELECT customerNumber, COUNT(*) FROM orders GROUP BY customerNumber;"},
]

messages = make_few_shot_messages(schema=schema_text, exemplars=exemplars, nlq=nlq)
print("number of messages:", len(messages))
print()
for message in messages:
    role = message["role"].upper()
    text = message["content"]
    if len(text) > 140:
        text = text[:140] + "..."
    print(f"[{role}] {text}")
    print()


## 3) Clean Noisy Model Output
Real models often add extra text. The pipeline extracts the first SQL statement and removes unsafe output.

In [ ]:
raw_output = """Sure, here is the SQL:

SELECT c.country, COUNT(*) AS order_count
FROM customers c
JOIN orders o ON c.customerNumber = o.customerNumber
GROUP BY c.country
ORDER BY order_count DESC;

This returns the countries with the most orders.
"""

first_sql = extract_first_select(raw_output)
clean_sql, reason = clean_candidate_with_reason(raw_output)
final_sql = guarded_postprocess(clean_sql or "", nlq) if clean_sql else None

print("Raw output:")
print(raw_output)
print()
print("First SQL:")
print(first_sql)
print()
print("Guardrail reason:", reason)
print("Final SQL after post-processing:")
print(final_sql)


## 4) Validate SQL
Validation checks that the SQL is a `SELECT` statement and that the table names exist in the schema.

In [ ]:
good_sql = final_sql
bad_sql = "SELECT country FROM order_log;"

print("Good SQL:")
print(good_sql)
print(validate_sql(good_sql, schema_text))
print()
print("Bad SQL:")
print(bad_sql)
print(validate_sql(bad_sql, schema_text))


## 5) Full System in One Sentence

In the real notebooks, the next step is simple: once SQL passes cleaning and validation, the system runs it against the database and scores the result.